# Swarm-style clipboard (Bokeh)

This notebook mirrors **Examples 1–3** in `Clipboard_Tutorial_A.ipynb`, but uses **`SwarmClipboardBk`**
(`vdapseisutils.core.swarmmpl.bokeh`) — the same *SwarmClipboard*-style API as the matplotlib v3 clipboard.

Tutorial A uses the legacy matplotlib **`Clipboard`** factory; here we use constructor kwargs (`wave_settings`,
`spec_settings`) instead of **`set_wave`** / **`set_spectrogram`**, and **`set_alim`** / **`set_flim`** for y-limits on the
Bokeh figures.

Example waveform files live under **`data/waveforms/`** relative to the repository root.


In [ ]:
from pathlib import Path


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "pyproject.toml").is_file():
            return p
        p = p.parent
    raise RuntimeError("Run this notebook from inside the vdapseisutils repository (or a subfolder).")


REPO = repo_root()
print("Repository:", REPO)

from obspy import read, UTCDateTime
from bokeh.io import output_notebook, show
from bokeh.plotting import save
from bokeh.resources import CDN

from vdapseisutils.style import colors as vdap_colors
from vdapseisutils.core.swarmmpl.bokeh import SwarmClipboardBk

output_notebook()


### Example 1: Clipboard layout (waveform + spectrogram)

Read Gareloi miniSEED, slice to a 10-minute window, default waveform + spectrogram stack per trace. Show inline and
write standalone HTML (Bokeh CDN resources).


In [ ]:
path = REPO / "data/waveforms/gareloi_test_data_20220710-010000.mseed"
if not path.is_file():
    raise FileNotFoundError(f"Missing example data: {path}")

st = read(str(path))
st = st.slice(UTCDateTime("2022/07/10 01:30:00"), UTCDateTime("2022/07/10 01:39:59.999"))
print(st)

cb = SwarmClipboardBk(
    st,
    mode="wg",
    tick_type="absolute",
    sync_waves=True,
    wave_settings={"color": "k"},
    spec_settings={"overlap": 0.86, "cmap": vdap_colors.viridis_u},
)
show(cb.layout)

out_html = REPO / "gallery/SwarmMPL/clipboard_tutorial_bokeh_example1.html"
save(cb.layout, filename=str(out_html), title="Gareloi — Bokeh clipboard example 1", resources=CDN)
print("Wrote", out_html)


### Example 2: “Pensive-like” options (filter + markers + limits)

Band-pass filter the same window, then draw vertical lines at selected UTC times and set waveform / spectrogram y-limits.


In [ ]:
path = REPO / "data/waveforms/gareloi_test_data_20220710-010000.mseed"
st = read(str(path))
st = st.slice(UTCDateTime("2022/07/10 01:30:00"), UTCDateTime("2022/07/10 01:39:59.999"))
st.filter("bandpass", freqmin=1.0, freqmax=10.0)
print(st)

cb2 = SwarmClipboardBk(
    st,
    mode="wg",
    tick_type="absolute",
    sync_waves=True,
    wave_settings={"color": "k"},
    spec_settings={"overlap": 0.86, "cmap": vdap_colors.viridis_u},
)
cb2.axvline("2022/07/10 01:30:15")
cb2.axvline(
    [
        "2022/07/10 01:32:08",
        "2022/07/10 01:33:52",
        "2022/07/10 01:34:54",
        "2022/07/10 01:36:26",
        "2022/07/10 01:37:59",
        "2022/07/10 01:39:48",
    ],
    color="red",
)
cb2.set_alim([-1000, 1000])
cb2.set_flim([0.1, 10.0])
show(cb2.layout)

out2 = REPO / "gallery/SwarmMPL/clipboard_tutorial_bokeh_example2.html"
save(cb2.layout, filename=str(out2), title="Gareloi — Bokeh clipboard example 2", resources=CDN)
print("Wrote", out2)


### Example 3: Relative time, independent panels, scroll

Augustine FI miniSEED: three traces on one channel, **`sync_waves=False`**, **`tick_type='relative'`**, then
**`scroll_traces`** with the same **`idx`** / **`seconds`** lists as Tutorial A.


In [ ]:
path = REPO / "data/waveforms/Augustine_test_data_FI.mseed"
if not path.is_file():
    raise FileNotFoundError(f"Missing example data: {path}")

st3 = read(str(path))
print(st3)

cb3 = SwarmClipboardBk(
    st3,
    mode="w",
    sync_waves=False,
    tick_type="relative",
    wave_settings={"color": "k"},
)
cb3.scroll_traces(idx=[0, 1, 2], seconds=[-0.5, 0.25, -0.25])
show(cb3.layout)

out3 = REPO / "gallery/SwarmMPL/clipboard_tutorial_bokeh_example3.html"
save(cb3.layout, filename=str(out3), title="Augustine — Bokeh clipboard example 3", resources=CDN)
print("Wrote", out3)
